# Day-Ahead 24-Hour Forecasting at 15-Minute Granularity (TFT)

This notebook performs **multi-step forecasting** for electricity price at **15-minute resolution**.

- Horizon: **24 hours = 96 steps**
- Uses past unknowns (price/demand) and known future covariates (weather + calendar).

## 1) Install dependencies (only if needed)
Run once if imports fail, then restart kernel.

In [1]:
%pip install -q pytorch-forecasting lightning holidays plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import warnings
from datetime import datetime
from pathlib import Path

import holidays
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

warnings.filterwarnings("ignore")
pl.seed_everything(42, workers=True)
torch.set_float32_matmul_precision("medium")

Seed set to 42


In [3]:
PROJECT_ROOT = Path.cwd().parent.parent.parent.parent
PRICE_PATH = PROJECT_ROOT / "data" / "price_data.csv"
WEATHER_PATH = PROJECT_ROOT / "data" / "historical_hourly_2023_2025.csv"
OUTPUT_PATH = PROJECT_ROOT / "Prediction" / "tft_day_ahead_24h_15min.csv"
WEATHER_DAILY = PROJECT_ROOT / "data" / "historical_daily_2023_2025.csv"

# 15-minute base series
price_df = pd.read_csv(PRICE_PATH)
ts_col = "timestamp" if "timestamp" in price_df.columns else "Timestamp"
price_df[ts_col] = pd.to_datetime(price_df[ts_col], utc=True, errors="coerce").dt.tz_convert(None)
price_df = price_df.rename(columns={ts_col: "Timestamp"})
price_df = price_df.sort_values("Timestamp").drop_duplicates("Timestamp")

base_cols = [c for c in ["Timestamp", "price", "demand_itsdo", "demand_indo", "demand_inddem", "demand_forecast", "wind_generation", "solar_generation", "margin_daily_forecast"] if c in price_df.columns]
price_df = price_df[base_cols]

full_index = pd.date_range(price_df["Timestamp"].min(), price_df["Timestamp"].max(), freq="15min")
price_df = price_df.set_index("Timestamp").reindex(full_index).rename_axis("Timestamp").reset_index()

# Hourly weather converted to smooth 15-minute weather by time interpolation
weather_df = pd.read_csv(WEATHER_PATH)
w_ts = "Timestamp" if "Timestamp" in weather_df.columns else "timestamp_utc"
weather_df[w_ts] = pd.to_datetime(weather_df[w_ts], utc=True, errors="coerce").dt.tz_convert(None)
weather_df = weather_df.rename(columns={w_ts: "WeatherTimestamp"}).sort_values("WeatherTimestamp")

hourly_keep = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "precipitation", "rain", "snowfall",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "shortwave_radiation", "direct_radiation",
    "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m",
    "surface_pressure", "weather_code"
]
hourly_keep = [c for c in hourly_keep if c in weather_df.columns]
weather_df = weather_df[["WeatherTimestamp"] + hourly_keep]

weather_df = weather_df.set_index("WeatherTimestamp")
weather_15min_index = pd.date_range(price_df["Timestamp"].min(), price_df["Timestamp"].max(), freq="15min")
weather_15min = weather_df.reindex(weather_15min_index)

numeric_weather_cols = [c for c in weather_15min.columns if c != "weather_code"]
if numeric_weather_cols:
    weather_15min[numeric_weather_cols] = weather_15min[numeric_weather_cols].interpolate(
        method="time", limit_direction="both"
    )

if "weather_code" in weather_15min.columns:
    weather_15min["weather_code"] = weather_15min["weather_code"].ffill().bfill()

weather_15min = weather_15min.reset_index().rename(columns={"index": "Timestamp"})
df = price_df.merge(weather_15min, on="Timestamp", how="left")

# Daily weather/sun features -> 15-minute features (ETL-style Solar_intensity + daily numeric vars)
daily_df = pd.read_csv(WEATHER_DAILY)
d_ts = "date_utc" if "date_utc" in daily_df.columns else "Timestamp"
daily_df[d_ts] = pd.to_datetime(daily_df[d_ts], utc=True, errors="coerce").dt.tz_convert(None)
daily_df = daily_df.rename(columns={d_ts: "Timestamp"}).sort_values("Timestamp")

if {"sunrise", "daylight_duration"}.issubset(daily_df.columns):
    sunrise = pd.to_datetime(daily_df["sunrise"], errors="coerce").dt.tz_localize(None)
    day_length_hours = daily_df["daylight_duration"] / 3600.0
    safe_day_length = day_length_hours.replace(0, np.nan)
    solar_noon_dt = sunrise + pd.to_timedelta(day_length_hours / 2, unit="h")
    hours = (daily_df["Timestamp"] - solar_noon_dt) / np.timedelta64(1, "h")
    daily_df["Solar_intensity"] = np.maximum(0, np.cos(hours * np.pi / safe_day_length)).fillna(0.0)

excluded_daily_cols = {"Timestamp", "sunrise", "sunset", "daylight_duration", "sunshine_duration"}
daily_numeric = [c for c in daily_df.columns if c not in excluded_daily_cols and pd.api.types.is_numeric_dtype(daily_df[c])]

daily_features = daily_df[["Timestamp"] + daily_numeric].drop_duplicates("Timestamp")
daily_features = daily_features.set_index("Timestamp")
daily_15min = daily_features.reindex(weather_15min_index).ffill().bfill().reset_index()
daily_15min = daily_15min.rename(columns={"index": "Timestamp"})

df = df.merge(daily_15min, on="Timestamp", how="left")

# Fill numeric gaps (demand / residual missing values)
for col in df.columns:
    if col == "Timestamp":
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].interpolate(limit_direction="both").ffill().bfill()

# filter the dataset to 2025
df = df[df["Timestamp"]> "2025-01-01"]

df = df.dropna(subset=["Timestamp", "price"]).sort_values("Timestamp").reset_index(drop=True)


print(f"Rows: {len(df):,}")
print(f"Range: {df['Timestamp'].min()} -> {df['Timestamp'].max()}")
print("Daily-derived features:", [c for c in ["Solar_intensity"] + daily_numeric if c in df.columns])
display(df.head(3))

Rows: 40,571
Range: 2025-01-01 00:15:00 -> 2026-02-27 14:45:00
Daily-derived features: ['Solar_intensity', 'shortwave_radiation_sum', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'precipitation_hours', 'temperature_2m_max', 'temperature_2m_min', 'Solar_intensity']


,Timestamp,price,demand_itsdo,demand_indo,demand_inddem,demand_forecast,wind_generation,solar_generation,margin_daily_forecast,temperature_2m,...,weather_code_x,weather_code_y,shortwave_radiation_sum,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,temperature_2m_max,temperature_2m_min,Solar_intensity
0,2025-01-01 00:15:00,9.07,26483.0,20949.0,-173.0,20900.0,20557.559,0.0,19370.0,6.4875,...,61.0,61.0,1.57,10.400001,10.400001,0.0,17.0,6.6,-2.85,0.492076
1,2025-01-01 00:30:00,23.51,26474.0,21129.0,-2338.0,20900.0,20108.433,0.0,19370.0,6.5250,...,61.0,61.0,1.57,10.400001,10.400001,0.0,17.0,6.6,-2.85,0.492076
2,2025-01-01 00:45:00,23.51,26474.0,21129.0,-2338.0,20900.0,20108.433,0.0,19370.0,6.5625,...,61.0,61.0,1.57,10.400001,10.400001,0.0,17.0,6.6,-2.85,0.492076


In [4]:
# Patch known bad price segment before train/val/test split
# Jan 8 between 11:15 and 21:00 where price > 200
if "price" not in df.columns:
    raise KeyError("Expected 'price' column in df")

if "price_raw" not in df.columns:
    df["price_raw"] = df["price"]

if "is_price_patched" not in df.columns:
    df["is_price_patched"] = False

target_year = int(df["Timestamp"].dt.year.mode().iloc[0])
target_day = pd.Timestamp(f"{target_year}-01-08").date()
start_time = pd.to_datetime("11:15").time()
end_time = pd.to_datetime("21:00").time()

window_mask = (
    (df["Timestamp"].dt.date == target_day)
    & (df["Timestamp"].dt.time >= start_time)
    & (df["Timestamp"].dt.time <= end_time)
)
bad_mask = window_mask & (df["price"] > 200)

ref_start = pd.Timestamp(f"{target_year}-01-01").date()
ref_end = pd.Timestamp(f"{target_year}-01-15").date()
reference_mask = (
    (df["Timestamp"].dt.date >= ref_start)
    & (df["Timestamp"].dt.date <= ref_end)
    & (df["Timestamp"].dt.date != target_day)
    & (df["price"] <= 200)
)

slot_median = (
    df.loc[reference_mask]
    .groupby(df.loc[reference_mask, "Timestamp"].dt.time)["price"]
    .median()
)

replacement_values = df.loc[bad_mask, "Timestamp"].dt.time.map(slot_median)
fallback_median = float(df.loc[reference_mask, "price"].median()) if reference_mask.any() else float(df["price"].median())
replacement_values = replacement_values.fillna(fallback_median)

df.loc[bad_mask, "price"] = replacement_values.values
df.loc[bad_mask, "is_price_patched"] = True

print(f"Patched rows: {int(bad_mask.sum())}")
print(f"Patch date: {target_day}, window: {start_time} to {end_time}")

Patched rows: 36
Patch date: 2025-01-08, window: 11:15:00 to 21:00:00


In [5]:
# 15-minute calendar features (known in the future)

uk_holidays = holidays.CountryHoliday("UK")
df["hour"] = df["Timestamp"].dt.hour.astype(str)
df["quarter_hour"] = (df["Timestamp"].dt.minute // 15).astype(str)
df["quarter_of_day"] = ((df["Timestamp"].dt.hour * 60 + df["Timestamp"].dt.minute) // 15).astype(str)
df["day_of_week"] = df["Timestamp"].dt.dayofweek.astype(str)
df["is_weekend"] = (df["Timestamp"].dt.dayofweek >= 5).astype(str)
df["is_holiday"] = df["Timestamp"].dt.date.isin(uk_holidays).astype(str)
df["is_working_day"] = ((df["is_weekend"] == "False") & (df["is_holiday"] == "False")).astype(str)

# Cyclical known reals
minute_of_day = df["Timestamp"].dt.hour * 60 + df["Timestamp"].dt.minute
df["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440.0)
df["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440.0)
df["dow_sin"] = np.sin(2 * np.pi * df["Timestamp"].dt.dayofweek / 7.0)
df["dow_cos"] = np.cos(2 * np.pi * df["Timestamp"].dt.dayofweek / 7.0)

# Time series ids
df["series_id"] = "GB"
df["time_idx"] = np.arange(len(df), dtype=np.int64)

MAX_PREDICTION_LENGTH = 96   # 24h * 4 steps/hour
MAX_ENCODER_LENGTH = 288     # 72h lookback at 15-min

known_weather = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "precipitation", "rain", "snowfall",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "shortwave_radiation", "direct_radiation",
    "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m",
    "surface_pressure", "Solar_intensity"
]
price_known_reals = ["demand_itsdo","demand_indo","demand_inddem","demand_forecast","wind_generation","solar_generation","margin_daily_forecast"]
daily_known_reals = [c for c in daily_numeric if c in df.columns and c != "price"] if "daily_numeric" in globals() else []
known_reals = [c for c in known_weather if c in df.columns] + daily_known_reals + ["tod_sin", "tod_cos", "dow_sin", "dow_cos"] + price_known_reals
known_reals = list(dict.fromkeys(known_reals))

known_categoricals = [
    c
    for c in ["hour", "quarter_hour", "quarter_of_day", "day_of_week", "is_weekend", "is_holiday", "is_working_day"]
    if c in df.columns
]
unknown_reals = [c for c in ["price"] if c in df.columns]

test_steps = 7 * 24 * 4
val_steps = 7 * 24 * 4
train_cutoff = df.time_idx.max() - (val_steps + test_steps + MAX_PREDICTION_LENGTH)
val_start = train_cutoff + 1
test_start = df.time_idx.max() - (test_steps + MAX_PREDICTION_LENGTH) + 1

print("Known real covariates:", known_reals)
print("Known categorical covariates:", known_categoricals)
print("Unknown real covariates:", unknown_reals)
print(f"train_cutoff={train_cutoff}, val_start={val_start}, test_start={test_start}")

Known real covariates: ['temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'shortwave_radiation', 'direct_radiation', 'wind_speed_10m', 'wind_gusts_10m', 'wind_direction_10m', 'surface_pressure', 'Solar_intensity', 'shortwave_radiation_sum', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'precipitation_hours', 'temperature_2m_max', 'temperature_2m_min', 'tod_sin', 'tod_cos', 'dow_sin', 'dow_cos', 'demand_itsdo', 'demand_indo', 'demand_inddem', 'demand_forecast', 'wind_generation', 'solar_generation', 'margin_daily_forecast']
Known categorical covariates: ['hour', 'quarter_hour', 'quarter_of_day', 'day_of_week', 'is_weekend', 'is_holiday', 'is_working_day']
Unknown real covariates: ['price']
train_cutoff=39130, val_start=39131, test_start=39803


In [6]:
training = TimeSeriesDataSet(
    df[df.time_idx <= train_cutoff],
    time_idx="time_idx",
    target="price",
    group_ids=["series_id"],
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    time_varying_known_reals=known_reals,
    time_varying_known_categoricals=known_categoricals,
    time_varying_unknown_reals=unknown_reals,
    target_normalizer=GroupNormalizer(groups=["series_id"]),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training, df, min_prediction_idx=val_start, stop_randomization=True
)
test_set = TimeSeriesDataSet.from_dataset(
    training, df, min_prediction_idx=test_start, stop_randomization=True
)

batch_size = 256
train_loader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_loader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=0)
test_loader = test_set.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

print(f"Train windows: {len(training):,} | Val windows: {len(validation):,} | Test windows: {len(test_set):,}")

Train windows: 38,748 | Val windows: 1,345 | Test windows: 673


In [7]:
early_stop = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=8, mode="min")
checkpoint_dir = PROJECT_ROOT / "models" / "price_prediction" / "tft_day_ahead_15min"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_cb = ModelCheckpoint(
    dirpath=checkpoint_dir,
    filename="tft-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    save_last=True,
 )

trainer = pl.Trainer(
    max_epochs=40,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop, checkpoint_cb],
    enable_checkpointing=True,
    logger=False
)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    output_size=7,
    loss=QuantileLoss(),
    reduce_on_plateau_patience=4
)

print(f"Model params: {tft.size()/1e3:.1f}k")
trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)

BEST_CHECKPOINT_PATH = checkpoint_cb.best_model_path
LAST_CHECKPOINT_PATH = checkpoint_cb.last_model_path
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Last checkpoint:", LAST_CHECKPOINT_PATH)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Model params: 244.7k


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  2.3 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  1.3 K │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  5.7 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 93.7 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 91.2 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    231 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 244 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 244 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 1328                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Best checkpoint: C:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min\tft-epoch=02-val_loss=5.7177.ckpt
Last checkpoint: C:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min\last-v4.ckpt


In [8]:
# Reload model later from checkpoint (run this in a new session after imports)
CKPT_TO_LOAD = BEST_CHECKPOINT_PATH if BEST_CHECKPOINT_PATH else LAST_CHECKPOINT_PATH
if not CKPT_TO_LOAD:
    raise ValueError("No checkpoint path found. Train first or set CKPT_TO_LOAD manually.")

loaded_tft = TemporalFusionTransformer.load_from_checkpoint(CKPT_TO_LOAD)
loaded_tft.eval()
print("Loaded checkpoint:", CKPT_TO_LOAD)

Loaded checkpoint: C:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min\tft-epoch=02-val_loss=5.7177.ckpt


In [9]:
# Save run metadata for reproducible inferencex
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
metadata_path = checkpoint_dir / f"run_metadata_{run_id}.json"
latest_metadata_path = checkpoint_dir / "latest_run_metadata.json"

run_metadata = {
    "run_id": run_id,
    "project_root": str(PROJECT_ROOT),
    "checkpoint_dir": str(checkpoint_dir),
    "best_checkpoint_path": BEST_CHECKPOINT_PATH,
    "last_checkpoint_path": LAST_CHECKPOINT_PATH,
    "model_class": "TemporalFusionTransformer",
    "max_prediction_length": int(MAX_PREDICTION_LENGTH),
    "max_encoder_length": int(MAX_ENCODER_LENGTH),
    "known_reals": known_reals,
    "known_categoricals": known_categoricals,
    "unknown_reals": unknown_reals,
    "train_cutoff": int(train_cutoff),
    "val_start": int(val_start),
    "test_start": int(test_start),
    "timestamp_min": str(df["Timestamp"].min()),
    "timestamp_max": str(df["Timestamp"].max()),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)
with open(latest_metadata_path, "w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)

print("Saved metadata:", metadata_path)
print("Updated latest metadata:", latest_metadata_path)

Saved metadata: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min\run_metadata_20260302_165110.json
Updated latest metadata: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\models\price_prediction\tft_day_ahead_15min\latest_run_metadata.json


In [10]:
def summarize_split(y_true: np.ndarray, y_pred: np.ndarray, mape_min_abs_actual: float = 10.0) -> dict:
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    diff = y_pred - y_true
    abs_diff = np.abs(diff)

    mae = float(np.mean(abs_diff))
    rmse = float(np.sqrt(np.mean(diff**2)))
    mape_raw = float(np.mean(abs_diff / np.clip(np.abs(y_true), 1e-6, None)) * 100)

    robust_mask = np.abs(y_true) >= mape_min_abs_actual
    mape_robust = float(np.mean(abs_diff[robust_mask] / np.abs(y_true[robust_mask])) * 100) if robust_mask.any() else np.nan

    smape = float(np.mean(2.0 * abs_diff / np.clip(np.abs(y_true) + np.abs(y_pred), 1e-6, None)) * 100)
    wape = float(np.sum(abs_diff) / np.clip(np.sum(np.abs(y_true)), 1e-6, None) * 100)

    actual_pct_change = (np.diff(y_true) / np.clip(np.abs(y_true[:-1]), 1e-6, None)) * 100 if len(y_true) > 1 else np.array([])
    pred_pct_change = (np.diff(y_pred) / np.clip(np.abs(y_pred[:-1]), 1e-6, None)) * 100 if len(y_pred) > 1 else np.array([])
    directional_accuracy = float(np.mean(np.sign(actual_pct_change) == np.sign(pred_pct_change))) if len(actual_pct_change) else np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        f"MAPE(|y|>={mape_min_abs_actual})(%)": mape_robust,
        "sMAPE(%)": smape,
        "WAPE(%)": wape,
        "MAPE_raw(%)": mape_raw,
        "DirectionalAccuracy": directional_accuracy,
    }

model_for_eval = globals().get("tft_loaded") or globals().get("loaded_tft")
if model_for_eval is None:
    raise ValueError("No loaded model found. Run Cell 9 (checkpoint load) first.")
model_for_eval = model_for_eval.cpu()
model_for_eval.eval()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

quantiles = [float(q) for q in model_for_eval.loss.quantiles] if hasattr(model_for_eval.loss, "quantiles") else []
median_idx = int(np.argmin(np.abs(np.array(quantiles) - 0.5))) if quantiles else 0

eval_batch_size = 16
MAX_TRAIN_WINDOWS = None
PLOT_WINDOWS_PER_SPLIT = None
FORCE_REEVALUATE = False
EXPORT_FULL_POINTS_CSV = False

train_eval_set = TimeSeriesDataSet.from_dataset(
    training,
    df[df.time_idx <= train_cutoff],
    stop_randomization=True
)
train_eval_loader = train_eval_set.to_dataloader(train=False, batch_size=eval_batch_size, num_workers=0)
val_eval_loader = validation.to_dataloader(train=False, batch_size=eval_batch_size, num_workers=0)
test_eval_loader = test_set.to_dataloader(train=False, batch_size=eval_batch_size, num_workers=0)

time_lookup = df[["time_idx", "Timestamp"]].drop_duplicates("time_idx").set_index("time_idx")["Timestamp"]

def evaluate_loader(loader, max_windows=None, plot_windows=None):
    y_true_batches = []
    y_pred_batches = []
    time_batches = []
    seen = 0

    with torch.no_grad():
        for x, y in loader:
            out = model_for_eval(x)
            pred_q = out["prediction"].detach().cpu().numpy()
            pred = pred_q[:, :, median_idx]

            true = y[0] if isinstance(y, (tuple, list)) else y
            true = true.detach().cpu().numpy()

            decoder_time_idx = x["decoder_time_idx"].detach().cpu().numpy()
            decoder_times = time_lookup.reindex(decoder_time_idx.reshape(-1)).to_numpy().reshape(decoder_time_idx.shape)

            y_true_batches.append(true)
            y_pred_batches.append(pred)
            time_batches.append(decoder_times)

            seen += true.shape[0]
            if max_windows is not None and seen >= max_windows:
                break

    y_true_all = np.concatenate(y_true_batches, axis=0) if y_true_batches else np.empty((0, MAX_PREDICTION_LENGTH))
    y_pred_all = np.concatenate(y_pred_batches, axis=0) if y_pred_batches else np.empty((0, MAX_PREDICTION_LENGTH))
    y_time_all = np.concatenate(time_batches, axis=0) if time_batches else np.empty((0, MAX_PREDICTION_LENGTH), dtype="datetime64[ns]")

    if plot_windows is not None and len(y_true_all) > plot_windows:
        y_true_plot = y_true_all[:plot_windows]
        y_pred_plot = y_pred_all[:plot_windows]
        y_time_plot = y_time_all[:plot_windows]
    else:
        y_true_plot = y_true_all
        y_pred_plot = y_pred_all
        y_time_plot = y_time_all

    return y_true_all, y_pred_all, y_true_plot, y_pred_plot, y_time_plot

def has_cached_split(prefix: str) -> bool:
    return all(
        name in globals() and getattr(globals()[name], "size", 0) != 0
        for name in [f"{prefix}_true", f"{prefix}_pred", f"{prefix}_time"]
    )

def load_or_eval_split(prefix: str, loader, max_windows=None, plot_windows=None):
    if (not FORCE_REEVALUATE) and has_cached_split(prefix):
        y_true_plot = globals()[f"{prefix}_true"]
        y_pred_plot = globals()[f"{prefix}_pred"]
        y_time_plot = globals()[f"{prefix}_time"]
        y_true_all = globals().get(f"{prefix}_true_all", y_true_plot)
        y_pred_all = globals().get(f"{prefix}_pred_all", y_pred_plot)
        return y_true_all, y_pred_all, y_true_plot, y_pred_plot, y_time_plot, True

    y_true_all, y_pred_all, y_true_plot, y_pred_plot, y_time_plot = evaluate_loader(
        loader, max_windows=max_windows, plot_windows=plot_windows
    )
    return y_true_all, y_pred_all, y_true_plot, y_pred_plot, y_time_plot, False

train_true_all, train_pred_all, train_true, train_pred, train_time, train_cached = load_or_eval_split(
    "train", train_eval_loader, max_windows=MAX_TRAIN_WINDOWS, plot_windows=PLOT_WINDOWS_PER_SPLIT
)
val_true_all, val_pred_all, val_true, val_pred, val_time, val_cached = load_or_eval_split(
    "val", val_eval_loader, max_windows=None, plot_windows=PLOT_WINDOWS_PER_SPLIT
)
test_true_all, test_pred_all, test_true, test_pred, test_time, test_cached = load_or_eval_split(
    "test", test_eval_loader, max_windows=None, plot_windows=PLOT_WINDOWS_PER_SPLIT
)

train_metrics = summarize_split(train_true_all, train_pred_all)
val_metrics = summarize_split(val_true_all, val_pred_all)
test_metrics = summarize_split(test_true_all, test_pred_all)

metrics_df = pd.DataFrame([
    {"Split": "Train", **train_metrics},
    {"Split": "Validation", **val_metrics},
    {"Split": "Test", **test_metrics},
]).set_index("Split")

mape_key = [c for c in metrics_df.columns if c.startswith("MAPE(|y|>=")][0]
overfit_check = pd.DataFrame({
    "Metric": ["MAE", "RMSE", mape_key, "sMAPE(%)", "WAPE(%)", "DirectionalAccuracy"],
    "Train": [train_metrics["MAE"], train_metrics["RMSE"], train_metrics[mape_key], train_metrics["sMAPE(%)"], train_metrics["WAPE(%)"], train_metrics["DirectionalAccuracy"]],
    "Validation": [val_metrics["MAE"], val_metrics["RMSE"], val_metrics[mape_key], val_metrics["sMAPE(%)"], val_metrics["WAPE(%)"], val_metrics["DirectionalAccuracy"]],
    "Test": [test_metrics["MAE"], test_metrics["RMSE"], test_metrics[mape_key], test_metrics["sMAPE(%)"], test_metrics["WAPE(%)"], test_metrics["DirectionalAccuracy"]],
})
overfit_check["Val-Train"] = overfit_check["Validation"] - overfit_check["Train"]
overfit_check["Test-Train"] = overfit_check["Test"] - overfit_check["Train"]

train_abs_err = np.abs((train_pred_all - train_true_all).reshape(-1))
train_err_diag = {
    "RMSE/MAE": float(train_metrics["RMSE"] / max(train_metrics["MAE"], 1e-9)),
    "AbsErr P95": float(np.percentile(train_abs_err, 95)),
    "AbsErr P99": float(np.percentile(train_abs_err, 99)),
    "AbsErr Max": float(np.max(train_abs_err)),
    "MAPE_raw(%)": float(train_metrics["MAPE_raw(%)"]),
    mape_key: float(train_metrics[mape_key]),
}

def build_split_points_df(split_name: str, y_true_split: np.ndarray, y_pred_split: np.ndarray, t_split: np.ndarray) -> pd.DataFrame:
    n_windows, horizon = y_true_split.shape
    out_df = pd.DataFrame({
        "split": split_name,
        "window_idx": np.repeat(np.arange(n_windows), horizon),
        "step_ahead": np.tile(np.arange(1, horizon + 1), n_windows),
        "timestamp": pd.to_datetime(t_split.reshape(-1)),
        "real_price": y_true_split.reshape(-1),
        "predicted_price": y_pred_split.reshape(-1),
    })
    out_df["abs_error"] = np.abs(out_df["predicted_price"] - out_df["real_price"])
    return out_df

train_points_df = build_split_points_df("Train", train_true, train_pred, train_time)
val_points_df = build_split_points_df("Validation", val_true, val_pred, val_time)
test_points_df = build_split_points_df("Test", test_true, test_pred, test_time)

def aggregate_by_timestamp(points_df: pd.DataFrame) -> pd.DataFrame:
    ts_df = (
        points_df.groupby(["split", "timestamp"], as_index=False)
        .agg(real_price=("real_price", "mean"),
             predicted_price=("predicted_price", "mean"),
             n_windows=("window_idx", "count"))
    )
    ts_df["abs_error"] = np.abs(ts_df["predicted_price"] - ts_df["real_price"])
    return ts_df

def build_window_summary(split_name: str, y_true_split: np.ndarray, y_pred_split: np.ndarray, t_split: np.ndarray) -> pd.DataFrame:
    rmse = np.sqrt(np.mean((y_pred_split - y_true_split) ** 2, axis=1))
    mae = np.mean(np.abs(y_pred_split - y_true_split), axis=1)
    return pd.DataFrame({
        "split": split_name,
        "window_idx": np.arange(len(y_true_split)),
        "start_time": pd.to_datetime(t_split[:, 0]),
        "end_time": pd.to_datetime(t_split[:, -1]),
        "window_mae": mae,
        "window_rmse": rmse,
    })

train_timestamp_df = aggregate_by_timestamp(train_points_df)
val_timestamp_df = aggregate_by_timestamp(val_points_df)
test_timestamp_df = aggregate_by_timestamp(test_points_df)
all_timestamp_df = pd.concat([train_timestamp_df, val_timestamp_df, test_timestamp_df], ignore_index=True)

train_window_df = build_window_summary("Train", train_true, train_pred, train_time)
val_window_df = build_window_summary("Validation", val_true, val_pred, val_time)
test_window_df = build_window_summary("Test", test_true, test_pred, test_time)
all_window_df = pd.concat([train_window_df, val_window_df, test_window_df], ignore_index=True)

train_window_rmse = train_window_df["window_rmse"].to_numpy()
worst_idx = np.argsort(train_window_rmse)[-10:][::-1]
worst_table = train_window_df.iloc[worst_idx][["window_idx", "window_rmse", "start_time", "end_time"]].reset_index(drop=True)

plots_dir = PROJECT_ROOT / "Prediction" / "plot_exports"
plots_dir.mkdir(parents=True, exist_ok=True)

metrics_path = plots_dir / "metrics_by_split.csv"
overfit_path = plots_dir / "overfit_check.csv"
worst_table_path = plots_dir / "top10_worst_train_windows.csv"
diag_path = plots_dir / "train_error_diagnostics.csv"

train_timestamp_path = plots_dir / "train_predictions_vs_real_by_timestamp.csv"
val_timestamp_path = plots_dir / "validation_predictions_vs_real_by_timestamp.csv"
test_timestamp_path = plots_dir / "test_predictions_vs_real_by_timestamp.csv"
all_timestamp_path = plots_dir / "all_splits_predictions_vs_real_by_timestamp.csv"

train_window_path = plots_dir / "train_predictions_vs_real_by_window.csv"
val_window_path = plots_dir / "validation_predictions_vs_real_by_window.csv"
test_window_path = plots_dir / "test_predictions_vs_real_by_window.csv"
all_window_path = plots_dir / "all_splits_predictions_vs_real_by_window.csv"

if EXPORT_FULL_POINTS_CSV:
    train_points_path = plots_dir / "train_predictions_vs_real_all_points.csv"
    val_points_path = plots_dir / "validation_predictions_vs_real_all_points.csv"
    test_points_path = plots_dir / "test_predictions_vs_real_all_points.csv"
    all_points_path = plots_dir / "all_splits_predictions_vs_real_all_points.csv"

    all_points_df = pd.concat([train_points_df, val_points_df, test_points_df], ignore_index=True)
    train_points_df.to_csv(train_points_path, index=False)
    val_points_df.to_csv(val_points_path, index=False)
    test_points_df.to_csv(test_points_path, index=False)
    all_points_df.to_csv(all_points_path, index=False)

metrics_df.to_csv(metrics_path)
overfit_check.to_csv(overfit_path, index=False)
worst_table.to_csv(worst_table_path, index=False)
pd.DataFrame([train_err_diag]).to_csv(diag_path, index=False)

train_timestamp_df.to_csv(train_timestamp_path, index=False)
val_timestamp_df.to_csv(val_timestamp_path, index=False)
test_timestamp_df.to_csv(test_timestamp_path, index=False)
all_timestamp_df.to_csv(all_timestamp_path, index=False)

train_window_df.to_csv(train_window_path, index=False)
val_window_df.to_csv(val_window_path, index=False)
test_window_df.to_csv(test_window_path, index=False)
all_window_df.to_csv(all_window_path, index=False)

worst_fig = make_subplots(
    rows=10, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.01,
    subplot_titles=[f"Window #{int(i)} | RMSE={train_window_rmse[int(i)]:.2f}" for i in worst_idx],
)

for r, i in enumerate(worst_idx, start=1):
    i = int(i)
    t = pd.to_datetime(train_time[i])
    worst_fig.add_trace(
        go.Scatter(x=t, y=train_true[i], mode="lines", name="Real", line=dict(width=1.8), showlegend=(r == 1)),
        row=r, col=1
    )
    worst_fig.add_trace(
        go.Scatter(x=t, y=train_pred[i], mode="lines", name="Predicted", line=dict(width=1.8, dash="dash"), showlegend=(r == 1)),
        row=r, col=1
    )
    worst_fig.update_yaxes(title_text="Price", row=r, col=1)

worst_fig.update_xaxes(title_text="Timestamp", row=10, col=1)
worst_fig.update_layout(
    title="Worst 10 training windows (highest RMSE)",
    height=2400,
    hovermode="x unified",
    legend_title="Series",
)

worst_html_path = plots_dir / "worst10_train_windows.html"
worst_fig.write_html(str(worst_html_path), include_plotlyjs="cdn", full_html=True)

print(f"Model used for evaluation: {type(model_for_eval).__name__}")
print(f"Median quantile index used: {median_idx} | quantile={quantiles[median_idx] if quantiles else 'N/A'}")
print(f"Evaluation batch size: {eval_batch_size} (CPU inference)")
print(f"Train windows evaluated: {len(train_true_all):,} (all available)")
print(f"Used cached splits -> train={train_cached}, val={val_cached}, test={test_cached}")
print(f"Saved: {metrics_path}")
print(f"Saved: {overfit_path}")
print(f"Saved: {diag_path}")
print(f"Saved: {worst_table_path}")
print(f"Saved: {train_timestamp_path}")
print(f"Saved: {val_timestamp_path}")
print(f"Saved: {test_timestamp_path}")
print(f"Saved: {all_timestamp_path}")
print(f"Saved: {train_window_path}")
print(f"Saved: {val_window_path}")
print(f"Saved: {test_window_path}")
print(f"Saved: {all_window_path}")
if EXPORT_FULL_POINTS_CSV:
    print(f"Saved: {train_points_path}")
    print(f"Saved: {val_points_path}")
    print(f"Saved: {test_points_path}")
    print(f"Saved: {all_points_path}")
print(f"Saved: {worst_html_path}")

Model used for evaluation: TemporalFusionTransformer
Median quantile index used: 3 | quantile=0.5
Evaluation batch size: 16 (CPU inference)
Train windows evaluated: 38,748 (all available)
Used cached splits -> train=False, val=False, test=False
Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\metrics_by_split.csv
Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\overfit_check.csv
Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\train_error_diagnostics.csv
Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\top10_worst_train_windows.csv
Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\train_predictions_vs_real_by_timestamp.csv


In [11]:
# Predict next 24h at 15-minute resolution (96 steps)
predict_ds = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)
predict_loader = predict_ds.to_dataloader(train=False, batch_size=1, num_workers=0)
raw_next, _, *_ = tft.predict(predict_loader, mode="raw", return_x=True)

next_96_pred = raw_next["prediction"][0, :, 3].detach().cpu().numpy()
last_ts = df["Timestamp"].max()
future_ts = pd.date_range(last_ts + pd.Timedelta(minutes=15), periods=96, freq="15min")

forecast_df = pd.DataFrame({
    "Timestamp": future_ts,
    "predicted_price": next_96_pred
})

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
forecast_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
display(forecast_df.head())

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Saved: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\tft_day_ahead_24h_15min.csv


,Timestamp,predicted_price
0,2026-02-27 15:00:00,70.486603
1,2026-02-27 15:15:00,69.238892
2,2026-02-27 15:30:00,70.532486
3,2026-02-27 15:45:00,73.052589
4,2026-02-27 16:00:00,78.435638


In [12]:
# Export interactive slider plots with timestamps to standalone HTML files
def build_window_slider_plot(split_name: str, y_true_split: np.ndarray, y_pred_split: np.ndarray, t_split: np.ndarray, out_path: Path):
    n_windows = len(y_true_split)
    if n_windows == 0:
        print(f"No windows available for {split_name}.")
        return

    x0 = pd.to_datetime(t_split[0])
    fig = go.Figure(data=[
        go.Scatter(x=x0, y=y_true_split[0], mode="lines", name="Real", line=dict(width=2)),
        go.Scatter(x=x0, y=y_pred_split[0], mode="lines", name="Predicted", line=dict(width=2, dash="dash")),
    ])

    frames = []
    for idx in range(n_windows):
        x_vals = pd.to_datetime(t_split[idx])
        ts_start = pd.to_datetime(t_split[idx][0])
        ts_end = pd.to_datetime(t_split[idx][-1])
        frames.append(
            go.Frame(
                name=str(idx),
                data=[
                    go.Scatter(x=x_vals, y=y_true_split[idx]),
                    go.Scatter(x=x_vals, y=y_pred_split[idx]),
                ],
                layout=go.Layout(title_text=f"{split_name}: Window #{idx} ({ts_start} to {ts_end})"),
            )
        )
    fig.frames = frames

    steps = [
        dict(
            method="animate",
            args=[[str(idx)], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
            label=str(idx),
        )
        for idx in range(n_windows)
    ]

    first_start = pd.to_datetime(t_split[0][0])
    first_end = pd.to_datetime(t_split[0][-1])

    fig.update_layout(
        title=f"{split_name}: Window #0 ({first_start} to {first_end})",
        sliders=[dict(active=0, currentvalue={"prefix": "Window: "}, pad={"t": 45}, steps=steps)],
        hovermode="x unified",
        xaxis_title="Timestamp",
        yaxis_title="Price",
        height=520,
    )
    fig.update_xaxes(rangeslider=dict(visible=True))

    fig.write_html(str(out_path), include_plotlyjs="cdn", full_html=True)
    print(f"Saved {split_name} slider HTML: {out_path}")

plots_dir = PROJECT_ROOT / "Prediction" / "plot_exports"
plots_dir.mkdir(parents=True, exist_ok=True)

build_window_slider_plot("Train", train_true, train_pred, train_time, plots_dir / "train_slider_all_windows.html")
build_window_slider_plot("Validation", val_true, val_pred, val_time, plots_dir / "validation_slider_all_windows.html")
build_window_slider_plot("Test", test_true, test_pred, test_time, plots_dir / "test_slider_all_windows.html")

Saved Train slider HTML: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\train_slider_all_windows.html
Saved Validation slider HTML: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\validation_slider_all_windows.html
Saved Test slider HTML: c:\Users\rkale\OneDrive\Documentos\VStudio\GitHub\Gitlab\GroupProject\25-26_CE903-SP_team03\Prediction\plot_exports\test_slider_all_windows.html


## Production note
For true day-ahead use, replace decoder weather values with **forecast weather** for the next 96 time steps.